# Week 4 — Solid cores, moving borders (exploration + validation)

Exercise 4.13 on the [course page](https://sunelehmann.com/socialgraphs2026-web/weeks/week4.html#go-nuts). The production pipeline is
`scripts/week4_analyze.py` (→ `data/processed/week4.json`); this notebook imports it, re-derives the key numbers, and keeps the
**validation cells**:

1. our NMI vs `sklearn.metrics.normalized_mutual_info_score`
2. our modularity vs `networkx.community.modularity`
3. our Louvain phase 1 (from the course's description) on the karate club, its Q vs networkx
4. our link clustering on a toy network with a known answer, plus the partition-density check
5. our disparity filter vs the course's α table and the explorable's α = 0.2 readout
6. the in-browser computations (`src/js/week4_core.js`) vs the Python output

Unless stated otherwise: the **undirected, unweighted giant component** of the philosophers network (1,374 nodes, 9,139 links).

In [1]:
import sys, json, itertools, subprocess, random
from pathlib import Path
from collections import Counter, defaultdict
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'week4' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
import networkx as nx, numpy as np, pandas as pd
from sklearn.metrics import normalized_mutual_info_score
import netlib, week4_analyze as w4

nodes_df, edges_df, D, G, GC, m_nodes, MD, MG, MGC = w4.load()
ids = sorted(GC)
print(f'philosophers: {G.number_of_nodes()} nodes, {D.number_of_edges()} directed edges, {G.number_of_edges()} undirected links')
print(f'giant component: {GC.number_of_nodes()} nodes, {GC.number_of_edges()} links')
print(f'marvel: {MG.number_of_nodes()} nodes, giant {MGC.number_of_nodes()} / {MGC.number_of_edges()}')
J = json.load(open(ROOT / 'data/processed/week4.json', encoding='utf-8'))
print('week4.json generated', J['meta']['generatedAt'])

philosophers: 1444 nodes, 11135 directed edges, 9140 undirected links
giant component: 1374 nodes, 9139 links
marvel: 303 nodes, giant 277 / 1421
week4.json generated 2026-09-23T12:58:24.561529+00:00


## 1 · Validation: our NMI against sklearn

The course formula, NMI = 2I / (H₁ + H₂), is sklearn's default (`average_method='arithmetic'`). Checked on all 45 pairs of ten Louvain runs.

In [2]:
runs = [netlib.partition_to_labels(nx.community.louvain_communities(GC, weight=None, seed=s), ids) for s in range(10)]
pairs = list(itertools.combinations(range(10), 2))
diffs = [abs(netlib.nmi(runs[i], runs[j]) - normalized_mutual_info_score(runs[i], runs[j])) for i, j in pairs]
vals = [netlib.nmi(runs[i], runs[j]) for i, j in pairs]
print(f'max |ours - sklearn| over 45 pairs: {max(diffs):.2e}')
print('NMI seeds 0 vs 1:', round(netlib.nmi(runs[0], runs[1]), 3), '| range over 45 pairs:', round(min(vals), 3), '-', round(max(vals), 3))
assert max(diffs) < 1e-9

max |ours - sklearn| over 45 pairs: 4.44e-16
NMI seeds 0 vs 1: 0.758 | range over 45 pairs: 0.653 - 0.833


## 2 · Validation: our modularity against networkx

Same partitions, unweighted and weighted. The trap we fell into first: networkx's `louvain_communities` **and** `modularity` both default to `weight='weight'`, so on this graph you get weighted numbers unless you pass `weight=None`.

In [3]:
for s in range(3):
    lab = dict(zip(ids, runs[s]))
    parts = defaultdict(set)
    for v, c in lab.items():
        parts[c].add(v)
    for wt in (None, 'weight'):
        ours, ref = netlib.modularity(GC, lab, weight=wt), nx.community.modularity(GC, list(parts.values()), weight=wt)
        print(f'seed {s} weight={wt!s:6}: ours {ours:.6f}  networkx {ref:.6f}')
        assert abs(ours - ref) < 1e-9
P = nx.community.louvain_communities(GC, seed=0)  # default weight='weight' !
print('default (weighted) Louvain, seed 0:', len(P), 'communities, weighted Q', round(nx.community.modularity(GC, P), 3), '- not the unweighted 0.50')

seed 0 weight=None  : ours 0.503258  networkx 0.503258
seed 0 weight=weight: ours 0.551201  networkx 0.551201
seed 1 weight=None  : ours 0.507544  networkx 0.507544
seed 1 weight=weight: ours 0.559566  networkx 0.559566
seed 2 weight=None  : ours 0.506428  networkx 0.506428
seed 2 weight=weight: ours 0.554233  networkx 0.554233


default (weighted) Louvain, seed 0: 9 communities, weighted Q 0.546 - not the unweighted 0.50


## 3 · Validation: Louvain phase 1, implemented from the course's description

Every node starts alone; visit nodes in (seeded) random order; move each to the neighbouring community with the largest ΔQ if it beats staying; sweep until a whole sweep moves nobody. For a node *i* taken out of its community, joining community *C* gains k_{i,C}/m − Σ_tot(C)·k_i/(2m²). Run on the karate club (unweighted).

In [4]:
def louvain_phase1(G, seed=0):
    rng = random.Random(seed)
    m = G.number_of_edges()
    k = dict(G.degree())
    comm = {v: v for v in G}
    tot = {v: k[v] for v in G}
    moved, sweeps = True, 0
    while moved:
        moved, sweeps = False, sweeps + 1
        order = list(G)
        rng.shuffle(order)
        for v in order:
            old = comm[v]
            links = Counter(comm[u] for u in G[v] if u != v)
            tot[old] -= k[v]  # take v out of its community
            gain = lambda c: links.get(c, 0) / m - tot[c] * k[v] / (2 * m * m)
            best = max(sorted(links), key=gain, default=old)
            target = best if gain(best) > gain(old) + 1e-12 else old
            comm[v] = target
            tot[target] += k[v]
            moved |= target != old
    return comm, sweeps

K = nx.karate_club_graph()
for seed in range(3):
    comm, sweeps = louvain_phase1(K, seed)
    parts = defaultdict(set)
    for v, c in comm.items():
        parts[c].add(v)
    q_ours = netlib.modularity(K, comm)
    q_nx = nx.community.modularity(K, list(parts.values()), weight=None)
    print(f'seed {seed}: {len(parts)} communities after {sweeps} sweeps, Q ours {q_ours:.4f}, networkx {q_nx:.4f}')
    assert abs(q_ours - q_nx) < 1e-12
full = nx.community.louvain_communities(K, weight=None, seed=0)
print('networkx full Louvain (both phases):', len(full), 'communities, Q', round(nx.community.modularity(K, full, weight=None), 4))

seed 0: 5 communities after 4 sweeps, Q ours 0.3991, networkx 0.3991
seed 1: 6 communities after 4 sweeps, Q ours 0.3614, networkx 0.3614
seed 2: 8 communities after 6 sweeps, Q ours 0.3320, networkx 0.3320
networkx full Louvain (both phases): 4 communities, Q 0.4198


**Why a different partition can still be "correct".** Phase 1 alone stops at a local optimum: no *single* node can raise Q by moving, but merging whole groups might (that's phase 2). And the modularity landscape is full of near-ties (§3 of the course page), so different visiting orders reach different local optima with almost the same Q. The validation that matters is that *our* Q for *our* partition equals networkx's Q for the *same* partition, and it does, exactly.

## 4 · Validation: link clustering on a toy with a known answer

Exercise 4.8's drawing: triangles {A,B,C} and {C,D,E} plus the link A–F. By eye: one link community per triangle, C in both, A–F on its own.

In [5]:
T = nx.Graph([('A','B'),('B','C'),('A','C'),('C','D'),('D','E'),('C','E'),('A','F')])
E, lab, Dbest, cut, Dcheck, npairs, curve = w4.link_clustering(T)
groups = defaultdict(list)
for e, c in zip(E, lab):
    groups[c].append(e)
for g in groups.values():
    print(sorted(g))
print(f'max partition density D = {Dbest:.3f} (brute-force recomputation {Dcheck:.3f}) at similarity {cut:.3f}')
node_comms = defaultdict(set)
for (u, v), c in zip(E, lab):
    node_comms[u].add(c); node_comms[v].add(c)
print('C is in', len(node_comms['C']), 'link communities')

[('A', 'B'), ('A', 'C'), ('B', 'C')]
[('A', 'F')]
[('C', 'D'), ('C', 'E'), ('D', 'E')]
max partition density D = 0.857 (brute-force recomputation 0.857) at similarity 0.600
C is in 2 link communities


In [6]:
S = J['linkcomm']['summary']
print(f"philosophers: {S['links']} links, {S['linkPairs']} adjacent link pairs, best D {S['bestD']} (brute force {S['Dcheck']}),",
      f"{S['nAtLeast3']} communities of 3+ links  | course single cut: D = 0.054, 444")

philosophers: 9139 links, 362270 adjacent link pairs, best D 0.0537 (brute force 0.0537), 444 communities of 3+ links  | course single cut: D = 0.054, 444


## 5 · Validation: our disparity filter against the course table and the explorable

Keep a link if (1 − w/s)^(k−1) < α at either end. The explorable's α = 0.2 readout: 1,540 links, 950 philosophers, giant component 816.

In [7]:
course = {0.05: (292, 348, 116), 0.1: (649, 607, 419), 0.2: (1540, 950, 816), 0.3: (2549, 1111, 1052), 0.5: (5641, 1284, 1270)}
pv = w4.disparity_pvalues(GC)
for a, ref in course.items():
    H = w4.backbone_graph(pv, a)
    ours = (H.number_of_edges(), H.number_of_nodes(), max(map(len, nx.connected_components(H))))
    print(f'alpha {a}: ours {ours}  course {ref}  {"OK" if ours == ref else "MISMATCH"}')
    assert ours == ref
for t, ref in {2: 3240, 3: 1572, 4: 785}.items():
    n = sum(1 for *_, w in GC.edges(data='weight') if w >= t)
    print(f'w >= {t}: {n} links (course {ref})')
    assert n == ref
s = dict(GC.degree(weight='weight'))
print('Moses of Narbonne:', round((1 - 2 / s['Moses_of_Narbonne']) ** (GC.degree('Moses_of_Narbonne') - 1), 4), '(course: 0.16, kept)')

alpha 0.05: ours (292, 348, 116)  course (292, 348, 116)  OK
alpha 0.1: ours (649, 607, 419)  course (649, 607, 419)  OK
alpha 0.2: ours (1540, 950, 816)  course (1540, 950, 816)  OK
alpha 0.3: ours (2549, 1111, 1052)  course (2549, 1111, 1052)  OK
alpha 0.5: ours (5641, 1284, 1270)  course (5641, 1284, 1270)  OK
w >= 2: 3240 links (course 3240)
w >= 3: 1572 links (course 1572)
w >= 4: 785 links (course 785)
Moses of Narbonne: 0.1643 (course: 0.16, kept)


## 6 · Validation: the browser against Python

`scripts/week4_validate_browser.mjs` imports the page's own `src/js/week4_core.js` and recomputes the backbone at the course's α values plus 15 sampled from the sweep, the three thresholds, the stability score of 26 sampled philosophers, and NMI(seed 0, seed 1).

In [8]:
out = subprocess.run(['node', str(ROOT / 'scripts' / 'week4_validate_browser.mjs')], capture_output=True, text=True)
rep = json.loads(out.stdout)
print('checks:', rep['summary']['checks'], '| failures:', rep['summary']['failures'])
print(pd.DataFrame(rep['alphas']).head(8).to_string(index=False))
print(pd.DataFrame(rep['stability']).head(6).to_string(index=False))

checks: 50 | failures: []
 alpha            python           browser   ok
 0.001        [7, 13, 3]        [7, 13, 3] True
 0.034    [189, 249, 43]    [189, 249, 43] True
 0.067   [405, 438, 239]   [405, 438, 239] True
 0.100   [649, 607, 419]   [649, 607, 419] True
 0.133   [889, 749, 548]   [889, 749, 548] True
 0.166  [1138, 821, 656]  [1138, 821, 656] True
 0.199  [1527, 947, 814]  [1527, 947, 814] True
 0.232 [1847, 1026, 914] [1847, 1026, 914] True
                   id  python  browser   ok
            Aristotle   0.823   0.8234 True
       Abraham_Tucker   0.805   0.8045 True
Albinus_(philosopher)   0.896   0.8956 True
    Anton_Wilhelm_Amo   0.545   0.5446 True
      Benjamin_Peirce   0.882   0.8820 True
 Chaitanya_Mahaprabhu   0.967   0.9667 True


## 7 · Exploration: the numbers the post is built on (from week4.json)

In [9]:
L, NP, NM = J['louvain'], J['nulls']['philosophers'], J['nulls']['marvel']
print(pd.DataFrame({'seed': L['seeds'], 'communities': L['nComm'], 'Q': L['q']}).to_string(index=False))
print(f"\nconsensus: {L['consensusN']} communities, Q {L['consensusQ']}, converged after {L['consensusIters']} round(s)")
for nm_, d in (('philosophers', NP), ('marvel', NM)):
    print(f"{nm_}: real Q {d['realMean']} | config {d['config']['mean']}±{d['config']['sd']} (z {d['config']['z']}) | swap {d['swap']['mean']} | G(n,m) {d['gnm']['mean']}")
print('NMI of two seeds on the same shuffled network:', NP['configNmiBetweenSeeds'])

 seed  communities      Q
    0            9 0.5033
    1           10 0.5075
    2            9 0.5064
    3           10 0.4998
    4           10 0.5072
    5            8 0.5010
    6            8 0.5012
    7            8 0.4999
    8           10 0.5055
    9            8 0.5073

consensus: 8 communities, Q 0.5086, converged after 1 round(s)
philosophers: real Q 0.5039 | config 0.2291±0.0024 (z 112.9) | swap 0.2224 | G(n,m) 0.231
marvel: real Q 0.383 | config 0.2674±0.0046 (z 25.1) | swap 0.2475 | G(n,m) 0.2821
NMI of two seeds on the same shuffled network: [0.084, 0.092, 0.095, 0.071, 0.104]


In [10]:
name = dict(zip(J['nodes']['id'], J['nodes']['name']))
print('least stable (degree >= 15):')
for v in L['borderDwellers'][:10]:
    i = J['nodes']['id'].index(v)
    print(f"  {name[v]:<28} degree {J['nodes']['degree'][i]:>3}  stability {L['stability'][i]}")
print('\nAristotle by run:', L['aristotleByRun'])

least stable (degree >= 15):
  Émile Durkheim               degree  18  stability 0.298
  Lucretius                    degree  19  stability 0.366
  Giordano Bruno               degree  30  stability 0.369
  Emanuel Swedenborg           degree  27  stability 0.373
  Lucilio Vanini               degree  23  stability 0.379
  Julius Caesar Scaliger       degree  18  stability 0.379
  Gerolamo Cardano             degree  15  stability 0.379
  Franciscus Patricius         degree  34  stability 0.379
  Charles Darwin               degree  57  stability 0.426
  Marin Mersenne               degree  24  stability 0.454

Aristotle by run: ['Aristotle', 'Aristotle', 'Thomas Aquinas', 'Aristotle', 'Aristotle', 'Aristotle', 'Aristotle', 'Aristotle', 'Aristotle', 'Avicenna']


In [11]:
I = J['infomap']
print(f"Infomap: {I['nModules']} modules, saves {I['savings']:.1%} of the code length; on shuffles {I['null']['nModulesMin']}-{I['null']['nModulesMax']} modules saving {I['null']['savingsMean']:.2%}")
print('NMI vs Louvain consensus:', I['nmiVsConsensus'], '| directed:', I['directed'])
mods = {m['id']: m for m in I['modules']}
for s_ in I['splits']:
    print(' Louvain', s_['louvain'], 'split into', [(mods[p['module']]['label'], p['n'], mods[p['module']]['internalLinks']) for p in s_['parts']])

Infomap: 35 modules, saves 9.2% of the code length; on shuffles 14-38 modules saving 0.25%
NMI vs Louvain consensus: 0.74 | directed: {'nModules': 61, 'codelength': 7.9783, 'savings': 0.1097, 'nmiVsUndirected': 0.643, 'nmiVsConsensus': 0.587}
 Louvain 0 split into [('John Locke', 182, 934), ('René Descartes', 45, 139)]
 Louvain 1 split into [('Immanuel Kant', 184, 1101), ('Leo Tolstoy', 45, 150)]
 Louvain 5 split into [('Adi Shankara', 38, 78), ('The Buddha', 35, 77), ('Madhvacharya', 20, 37)]
 Louvain 6 split into [('Avicenna', 45, 151), ('Maimonides', 32, 99)]


### Reading five disputed philosophers (exercise 4.9's validation)

Philosophers Infomap puts in a module dominated by a different Louvain community. Below: the node file's one-line description, read alongside their Wikipedia pages.

In [12]:
desc = dict(zip(nodes_df.node_id, nodes_df.description))
cons_label = {r['id']: r['label'] for r in J['communities']['rows']}
deg = dict(GC.degree())
for v in sorted(I['disputed'], key=lambda v: -deg[v])[:5]:
    i = J['nodes']['id'].index(v)
    print(f"{name[v]} — Louvain: {cons_label[L['consensus'][i]]}'s; Infomap module around {mods[I['labels'][i]]['label']}")
    print('   ', str(desc[v])[:220], '\n')

Gottfried Wilhelm Leibniz — Louvain: René Descartes's; Infomap module around Immanuel Kant
    Gottfried Wilhelm Leibniz was a German polymath active as a mathematician, philosopher, scientist, and diplomat who is credited, alongside Isaac Newton, with the creation of calculus in addition to many other branches of 

Averroes — Louvain: Avicenna's; Infomap module around Thomas Aquinas
    Ibn Rushd, Latinized as Averroes, was an Andalusian polymath and jurist who was proficient in a variety of intellectual fields, including philosophy, theology, medicine, astronomy, physics, psychology, mathematics, neuro 

Martin Luther — Louvain: Thomas Aquinas's; Infomap module around Immanuel Kant
    Martin Luther was a German priest, theologian, author, hymnwriter, professor, and former Augustinian friar. 

Erasmus — Louvain: Thomas Aquinas's; Infomap module around Aristotle
    Desiderius Erasmus Roterodamus, commonly known in English as Erasmus of Rotterdam or simply Erasmus, was a Dutch humanis

**Our reading.** Each disagreement lines up with a real double context, which is what the prompt for 4.9 asks us to check:

- *Leibniz*: Louvain keeps him with Descartes (early-modern rationalism, calculus with Newton); Infomap's walker follows his heavy links into German philosophy (Wolff, Kant), which he founded as much as the Cartesian line.
- *Averroes*: Islamic philosophy for Louvain; the scholastic module for Infomap. Latin Averroism is a whole chapter of scholasticism.
- *Martin Luther*: Christian theology for Louvain; German philosophy for Infomap (the Reformation is where both histories start).
- *Erasmus*: Christian humanism for Louvain; the classical/antiquity module for Infomap. He edited the Church Fathers and the classics.
- *Franciscus Patricius*: antiquity for Louvain (a Platonist writing against Aristotle); early modern for Infomap (his new natural philosophy).

None of these is a mistake by either method: the philosopher genuinely belongs to two contexts, and a partition must pick one. That is the overlap argument of section 5, arriving from a different direction.

In [13]:
B = J['backbone']
print('break:', {k: B['break'][k] for k in ('alpha', 'secondLargest', 'giantAtBreak', 'alphaAbove', 'giantAbove')})
print('breaker:', B['break']['breaker']['node'], B['break']['breaker']['pieces'])
for h in B['hinges']:
    print(f"  below alpha {h['alpha']:<7} {h['a']} - {h['b']}: {h['group']['size']} detach ({', '.join(h['group']['top'][:3])}) sole bridge={h['soleBridge']}")

break: {'alpha': 0.045, 'secondLargest': 40, 'giantAtBreak': 61, 'alphaAbove': 0.048, 'giantAbove': 110}
breaker: Erasmus [61, 45, 1, 1]
  below alpha 0.2972  Gaudapada - The_Buddha: 24 detach (Adi_Shankara, Madhvacharya, Ramanuja) sole bridge=True
  below alpha 0.1696  Confucius - Voltaire: 22 detach (Confucius, Zhu_Xi, Laozi) sole bridge=True
  below alpha 0.1519  Ibn_Arabi - Ibn_Hazm: 613 detach (Aristotle, Plato, Immanuel_Kant) sole bridge=False
  below alpha 0.0916  Gottfried_Wilhelm_Leibniz - Wilhelm_Wundt: 10 detach (Albert_Einstein, Wilhelm_Wundt, Hermann_von_Helmholtz) sole bridge=True
  below alpha 0.081   Herbert_Spencer - William_James: 17 detach (William_James, Charles_Sanders_Peirce, John_Dewey) sole bridge=True
  below alpha 0.0685  Diogenes_Laertius - Epicurus: 10 detach (Diogenes_Laertius, Anaximander, Zeno_of_Citium) sole bridge=True
  below alpha 0.0518  David_Hume - Jean-Jacques_Rousseau: 34 detach (Immanuel_Kant, Georg_Wilhelm_Friedrich_Hegel, David_Hume) sole brid

## 8 · Girvan–Newman, on the karate club only

Full Girvan–Newman on the philosophers is hours of compute for a result Louvain gets in a second (course §2), so the illustration runs on the karate club: the first cut should be Mr. Hi (0) – member 31.

In [14]:
eb = nx.edge_betweenness_centrality(K, normalized=False)
top = sorted(eb.items(), key=lambda kv: -kv[1])[:2]
print('highest edge betweenness:', [(e, round(b, 1)) for e, b in top], '(course: 0-31 at 71.4, next 43.8)')
first = next(nx.community.girvan_newman(K))
print('first split sizes:', sorted(map(len, first), reverse=True))

highest edge betweenness: [((0, 31), 71.4), ((0, 6), 43.8)] (course: 0-31 at 71.4, next 43.8)
first split sizes: [19, 15]
